<a href="https://colab.research.google.com/github/aiportfoliorhea/ai-portfolio/blob/main/loanbot_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install fastapi uvicorn chromadb langchain pypdf sentence-transformers langchain-community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is inc

In [24]:
import chromadb
import langchain
import fastapi

In [9]:
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Create ChromaDB client and collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="loanbot")

print("ChromaDB collection created ✓")

ChromaDB collection created ✓


In [11]:
# 2. Create a simple test PDF to work with

sample_text = """
LOAN AGREEMENT

Section 1: Loan Terms
The borrower agrees to repay a principal amount of $500,000 at a fixed interest rate of 7.2% per annum over a period of 15 years. Monthly payments shall be $4,561.

Section 2: Prepayment
Borrower may prepay any amount without penalty after the first 12 months. Prepayment within the first 12 months incurs a fee of 2% of the remaining balance.

Section 3: Late Payment
Payments received more than 10 days after the due date will incur a late fee of $150 or 3% of the payment amount, whichever is greater.

Section 4: Default
Borrower is considered in default after 60 days of non-payment. Upon default, the full remaining balance becomes immediately due.
"""

# Save it as a text file
with open("sample_loan.txt", "w") as f:
    f.write(sample_text)

print("Sample loan doc created ✓")

Sample loan doc created ✓


In [21]:

with open("sample_loan.txt", "r") as f:
    raw_text = f.read()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = splitter.create_documents([raw_text])

In [22]:
print(f"Number of chunks {len(chunks)}")

Number of chunks 5


In [23]:
for i in range(len(chunks)):
    print(f"Chunk {i}: {chunks[i]}")

Chunk 0: page_content='LOAN AGREEMENT'
Chunk 1: page_content='Section 1: Loan Terms
The borrower agrees to repay a principal amount of $500,000 at a fixed interest rate of 7.2% per annum over a period of 15 years. Monthly payments shall be $4,561.'
Chunk 2: page_content='Section 2: Prepayment
Borrower may prepay any amount without penalty after the first 12 months. Prepayment within the first 12 months incurs a fee of 2% of the remaining balance.'
Chunk 3: page_content='Section 3: Late Payment
Payments received more than 10 days after the due date will incur a late fee of $150 or 3% of the payment amount, whichever is greater.'
Chunk 4: page_content='Section 4: Default
Borrower is considered in default after 60 days of non-payment. Upon default, the full remaining balance becomes immediately due.'


In [25]:
# 4. Store chunks in ChromaDB

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content],
        ids=[f"chunk_{i}"]
    )

print(f"Stored {len(chunks)} chunks in ChromaDB ✓")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 49.0MiB/s]


Stored 5 chunks in ChromaDB ✓


In [30]:
#  5. Query ChromaDB

results = collection.query(
    query_texts=["what happens if I pay late?"],
    n_results=2
)
print(results)

{'ids': [['chunk_3', 'chunk_4']], 'embeddings': None, 'documents': [['Section 3: Late Payment\nPayments received more than 10 days after the due date will incur a late fee of $150 or 3% of the payment amount, whichever is greater.', 'Section 4: Default\nBorrower is considered in default after 60 days of non-payment. Upon default, the full remaining balance becomes immediately due.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.5444183945655823, 1.2410978078842163]]}
